# Hyperparameter tuning for XGBoost

In [1]:
import time
from pathlib import Path
import pandas as pd
import sys
import wandb
import pickle

sys.path.insert(0, "../../")
from geometric_extraction_helper import GENERAL_KEYS, MATERIAL_KEYS, RAY_KEYS
from models_helper import ML_Models, ModelTrainer

from env_helper import get_env_var
from wandb_helper import get_sweep_params, log_ml_metrics, set_run_name, start_training

WANDB_PROJECT = get_env_var("WANDB_PROJECT", default="ebkph-classifier", allow_empty=True)
WANDB_ENTITY  = get_env_var("WANDB_ENTITY",  allow_empty=True)

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# load data
DATA_DIR = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed.parquet")
df_train_over = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed_oversampled.parquet")
df_val = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


In [ ]:
# set n for top n feature selection
top_n = 9

# get feature importance ranking from previous step
fi_df = pd.read_json("../4_2_train advanced ML models/feature_importance_xgboost.json", orient="records")
ordered_features = fi_df["feature"].tolist()
FEATURE_SUBSET = ordered_features[:top_n]
FEATURE_SUBSET
#ALL_FEATURE_KEYS = GENERAL_KEYS + MATERIAL_KEYS + RAY_KEYS


['geom_layer_count',
 'topo_unique_edge_count',
 'mat_belag',
 'mat_gips',
 'topo_euler_characteristic',
 'mat_kies',
 'mat_beton',
 'aabb_ratio_x_y',
 'geom_ratio_area_vol',
 'mat_zement',
 'aabb_len_x',
 'mat_metall',
 'mat_bekleidung',
 'mat_backstein',
 'mat_kunststein',
 'aabb_len_y',
 'geom_compactness',
 'aabb_len_z',
 'aabb_max_z',
 'aabb_ratio_z_y',
 'tfbb_extent_2',
 'mat_aluminium',
 'tfbb_ratio_12',
 'aabb_max_x',
 'aabb_ratio_z_x',
 'topo_connected_components',
 'topo_avg_face_area',
 'topo_face_count',
 'mat_werkstoff',
 'topo_vertex_edge_ratio',
 'tfbb_extent_1',
 'mat_allgemein',
 'topo_vertex_count',
 'tfbb_extent_0',
 'aabb_min_z']

In [ ]:
# get labels
label_cols = [c for c in df_train.columns if c.startswith("label_")]

# get data splits
X_train = df_train[FEATURE_SUBSET]
y_train = df_train[label_cols]
X_train_over = df_train_over[FEATURE_SUBSET]
y_train_over = df_train_over[label_cols]
X_val = df_val[FEATURE_SUBSET]
y_val = df_val[label_cols]

print(f"Train: {len(X_train)}, Train (oversampled): {len(X_train_over)}, Val: {len(X_val)}")
print(f"Labels: {label_cols}")

Train: 20594, Train (oversampled): 26465, Val: 9708
Labels: ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 1. Bayesian Hyperparameter Tuning for XGBoost

In [ ]:
# name of best model file
BEST_MODEL_PATH = "best_xgboost_model"

# first version of the hyperparameter search space
sweep_config_v1 = {
    "method": "bayes",
    "metric": {"name": "val/f1_macro", "goal": "maximize"},
    "parameters": {
        "max_depth":           {"min": 0,     "max": 10,   "distribution": "int_uniform"},
        "min_child_weight":    {"min": 0,     "max": 10,   "distribution": "int_uniform"},
        "gamma":               {"min": 0.0,   "max": 5.0,  "distribution": "uniform"},
        "reg_alpha":           {"min": 0.0,   "max": 10.0, "distribution": "uniform"},
        "reg_lambda":          {"min": 0.0,   "max": 5.0,  "distribution": "uniform"},
        "colsample_bytree":    {"min": 0.5,   "max": 1.0,  "distribution": "uniform"},
        "colsample_bylevel":   {"min": 0.5,   "max": 1.0,  "distribution": "uniform"},
        "subsample":           {"min": 0.3,   "max": 1.0,  "distribution": "uniform"},
        "learning_rate":       {"min": 0.005, "max": 0.3,  "distribution": "log_uniform_values"},
        "n_estimators":        {"values": [200, 500, 1000]},
        "training_oversample": {"values": [True, False]},
    }
}

best = {"score": -1}

def train_fn():
    with wandb.init() as run:
        wandb.log({"model_type": "xgboost"})
        wandb.log({"categories": "top_n_feature_selection"})
        set_run_name(run, config_params=["max_depth", "min_child_weight", "gamma", "reg_alpha", "reg_lambda", "colsample_bytree", "colsample_bylevel", "subsample", "learning_rate", "n_estimators", "training_oversample"])

        # get params and pop oversampling param
        all_params = get_sweep_params(run)
        oversample = all_params.pop("training_oversample", False)
        params = all_params

        # log count of features and all features used for training
        wandb.log({"num_features": len(X_train_over.columns if oversample else X_train.columns)})
        wandb.log({"feature_names": list(X_train_over.columns if oversample else X_train.columns)})

        # get training data (oversampled or not)
        X_tr = X_train_over if oversample else X_train
        y_tr = y_train_over if oversample else y_train

        pipeline = ML_Models.xgboost(**params)
        model = ModelTrainer(pipeline, label_cols)

        # start training and measure duration
        t0 = time.time()
        model.fit(X_tr, y_tr)
        train_duration_s = round(time.time() - t0, 2)

        log_ml_metrics(run, model.evaluate(X_tr, y_tr), split="train")
        val_metrics = model.evaluate(X_val, y_val)
        log_ml_metrics(run, val_metrics, split="val")
        wandb.log({"train_duration_s": train_duration_s})
        
        # save model if it's the best so far for the sweep
        val_f1 = val_metrics.loc["mean", "f1_macro"]
        if val_f1 > best["score"]:
            best["score"] = val_f1
            Path("./models").mkdir(exist_ok=True)
            current_name = f"./models/{BEST_MODEL_PATH}_{val_f1:.4f}.pkl"
            with open(current_name, "wb") as f:
                pickle.dump(model, f)
            print(f"New best model (val/f1_macro={val_f1:.4f}) saved to {current_name}")

start_training(
    train_fn,
    use_wandb=True,
    sweep_config=sweep_config_v1,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    sweep_count=1000,
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/lukasstoeckli/.netrc
wandb: Currently logged in as: luki-st (baa_ls) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: 1c18b3db
Sweep URL: https://wandb.ai/baa_ls/eBKPh_classifier/sweeps/1c18b3db


wandb: Agent Starting Run: zd6m1amu with config:
wandb: 	colsample_bylevel: 0.5989989300328045
wandb: 	colsample_bytree: 0.7439304254404464
wandb: 	gamma: 0.6400696699098735
wandb: 	learning_rate: 0.022438875553956007
wandb: 	max_depth: 10
wandb: 	min_child_weight: 8
wandb: 	n_estimators: 500
wandb: 	reg_alpha: 6.284917452349851
wandb: 	reg_lambda: 3.9068998392701944
wandb: 	subsample: 0.7575191002833602
wandb: 	training_oversample: True
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/lukasstoeckli/.netrc.
